# End-to-End Small Language Model Fine Tuning to Generate Manim Code
This notebook is a companion of chapter 3 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is about fine tuning a small language model, [GPT-2 small](https://huggingface.co/openai-community/gpt2) to generate Python [Manim](https://github.com/ManimCommunity/manim) code. Hardware acceleration (GPU) is needed.  
More details about the code can be found in the related book's chapter.

Install the missing dependencies in the Colab VM (only Optuna for hyperparameter search isn't available by default).

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 14.8 MB/s eta 0:00:00


### Data Preparation

Download the [Manim_Python dataset](https://huggingface.co/datasets/Edoh/manim_python) from the Hugging Face's Hub.

In [2]:
from datasets import load_dataset

dataset = load_dataset("Edoh/manim_python")

README.md:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/135k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/11.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/599 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/51 [00:00<?, ? examples/s]

Display some training sample.

In [3]:
print(dataset["train"][0])

{'instruction': "Create a new scene named 'MyScene'.", 'output': 'from manim import * class MyScene(Scene): def construct(self): pass'}


Download the GPT2 Small model's tokenizer.

In [4]:
from transformers import GPT2Tokenizer

model_name = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Set pad token to eos token for the model's tokenizer.

In [5]:
tokenizer.pad_token = tokenizer.eos_token

Define a custom function to preprocess the raw data. It concatenates the `instruction` and `output` columms of the dataset, tokenizes the concatenated text and set the lables same as the input_ids.

In [6]:
def preprocess_data(examples):
    inputs = [
        f"Instruction: {instr}\nOutput: {out}"
        for instr, out in zip(examples["instruction"], examples["output"])
    ]
    tokenized = tokenizer(inputs, truncation=True, max_length=512, padding="max_length")

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

Use the `preprocess_data` function to tokenize the training data. The original columns are also removed, as they aren't needed anymore for training purposes.

In [7]:
tokenized_datasets = dataset.map(preprocess_data,
                                 batched=True,
                                 remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

### Hyperparameter Tuning

Import the required classes.

In [8]:
from transformers import (
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

Define a custom function to init the model during the hyperparameter search. This way we will have a fresh model for each trial. The baseline model weights are pulled from the HF's Hub only during the first trial. Subsequents trials will use the cached weights.

In [9]:
def model_init():
    return GPT2LMHeadModel.from_pretrained(model_name, device_map='auto')

Set the training arguments.

In [10]:
training_args = TrainingArguments(
    output_dir="./gpt2-manim-python-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    report_to="none",
)

Split the training data, to reserve 10% for validation.

In [11]:
train_val_split = tokenized_datasets["train"].train_test_split(test_size=0.1)
tokenized_datasets["train"] = train_val_split["train"]
tokenized_datasets["validation"] = train_val_split["test"]

Create an instance of data collator for causal language modeling.

In [12]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Create an instance of Trainer. An early stopping callback is set too.

In [14]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Define the hyperparameter search space.

In [15]:
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [2, 4, 8]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 6),
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
        "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4]),
    }

Run the hyperparameter search. The Optuna backend is used.

In [16]:
trials_count = 3
best_run = trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    n_trials=trials_count,
    hp_space=hp_space,
    compute_objective=lambda metrics: metrics["eval_loss"],
)

[I 2026-08-08 15:32:36,198] A new study created in memory with name: no-name-4a0636ad-5447-4a3c-9156-91383dfdad3a


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.339695,0.431271
2,0.464295,0.258968
3,0.237715,0.218600
4,0.205337,0.204323
5,0.210387,0.204959


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
[I 2026-08-08 15:52:50,326] Trial 2 finished with value: 0.18040363490581512 and parameters: {'learning_rate': 0.00025543978503595057, 'per_device_train_batch_size': 2, 'weight_decay': 0.25162254422941965, 'num_train_epochs': 4, 'warmup_steps': 484, 'gradient_accumulation_steps': 1}. Best is trial 2 with value: 0.18040363490581512.


Display the best combination of hyperparmaters found.

In [17]:
print("Best hyperparameters found:", best_run.hyperparameters)

Best hyperparameters found: {'learning_rate': 0.00025543978503595057, 'per_device_train_batch_size': 2, 'weight_decay': 0.25162254422941965, 'num_train_epochs': 4, 'warmup_steps': 484, 'gradient_accumulation_steps': 1}


### Training

Configure the Trainer with the best hyperparameters.

In [18]:
for key, value in best_run.hyperparameters.items():
    setattr(training_args, key, value)

In [20]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Start the model fine-tuning.

In [21]:
trainer.train()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss
1,0.355537,0.256605
2,0.234750,0.304930
3,0.147284,0.197724
4,0.117684,0.180404


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=1080, training_loss=0.3373862513789424, metrics={'train_runtime': 392.6756, 'train_samples_per_second': 5.491, 'train_steps_per_second': 2.75, 'total_flos': 563345620992000.0, 'train_loss': 0.3373862513789424, 'epoch': 4.0})

Save the fine-tuned model and the tokenizer to disk.

In [23]:
trainer.save_model("./gpt2-manim-python-finetuned")
tokenizer.save_pretrained("./gpt2-manim-python-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-manim-python-finetuned/tokenizer_config.json',
 './gpt2-manim-python-finetuned/tokenizer.json')

### Test

Load the fine-tuned model from disk to GPU memory. Load also its companion tokenizer.

In [24]:
import torch

model_dir = "./gpt2-manim-python-finetuned"
tokenizer = GPT2Tokenizer.from_pretrained(model_dir)
model = GPT2LMHeadModel.from_pretrained(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

Define a custom function to do Manim code generation with the fine-tuned model and decode the output.

In [25]:
def generate_output(instruction, max_length=150, num_beams=5, temperature=0.7, top_p=0.9, repetition_penalty=1.2):
    """
    Generate output text given an instruction using beam search and nucleus sampling.

    Args:
        instruction (str): The input instruction prompt.
        max_length (int): Maximum length of generated sequence (including prompt).
        num_beams (int): Number of beams for beam search.
        temperature (float): Sampling temperature; lower is less random.
        top_p (float): Nucleus sampling probability threshold.
        repetition_penalty (float): Penalty for repeated tokens (>1.0 discourages repetition).

    Returns:
        str: Generated output text.
    """

    prompt = f"Instruction: {instruction}\nOutput:"

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    generated_ids = model.generate(
        input_ids,
        max_length=max_length,
        num_beams=num_beams,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        early_stopping=True,
        no_repeat_ngram_size=2,
    )

    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    output_start = generated_text.find("Output:")
    if output_start != -1:
        output_text = generated_text[output_start + len("Output:"):].strip()
    else:
        output_text = generated_text.strip()

    return output_text

Execute the fine-tuned model on all the samples in the test set and save the generated Manim code snippets (along with the corresponding prompt and ground truth) in a CSV file.

In [26]:
import csv

output_csv = "gpt2_manim_python_test_outputs.csv"
with open(output_csv, mode="w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["instruction", "reference_output", "generated_output"])
    writer.writeheader()

    for example in dataset['test']:
        instruction = example["instruction"]
        reference_output = example["output"]

        generated_output = generate_output(instruction)

        writer.writerow({
            "instruction": instruction,
            "reference_output": reference_output,
            "generated_output": generated_output,
        })

print(f"Inference complete. Results saved to {output_csv}")

Inference complete. Results saved to gpt2_manim_python_test_outputs.csv
